# Multiagent: specialist 팀 조율하기

Claude Managed Agents와 multi-agent coordinator 패턴을 사용하여 Northstar라는 가상 회사의 판매 제안서 작성을 자동화하겠습니다. 이 회사는 중견 기업의 운영 팀에 workflow 자동화 플랫폼을 판매합니다.

현재 영업 담당자는 잠재 고객별 맞춤 제안서를 만듭니다. 잠재 고객과 같은 업종의 기업이 보통 무엇을 우선하는지 조사하고, 수백 개의 사례 모음에서 관련 사례 두 개를 가져오고, 내부 규칙표로 가격을 계산한 뒤, 이를 두 페이지 문서로 구성합니다. 각 단계는 서로 다른 출처와 서로 다른 종류의 판단을 활용합니다.

이 작업을 위해 coordinator agent가 specialist 세 명을 실행하도록 하겠습니다. researcher는 web search로 잠재 고객의 업종에서 일반적인 내용을 찾습니다. librarian은 사례 모음을 읽고 가장 잘 맞는 두 개를 고릅니다. pricing modeler는 규칙 파일과 seat 수만 봅니다. coordinator는 이들의 순서를 정하고 제안서를 작성합니다.


## 1. client 설정하기

먼저 SDK를 설치하고 Anthropic client를 설정하겠습니다. multiagent 설정과 event 타입은 Managed Agents beta에 포함되어 있습니다.


In [ ]:
%%capture
%pip install -qU "anthropic>=0.121.0" python-dotenv

In [ ]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-opus-5")
ADVISOR_MODEL = os.environ.get("COOKBOOK_ADVISOR_MODEL", "claude-opus-5")
client = anthropic.Anthropic()

## 2. specialist subagent 세 명 정의하기

다음으로 팀원 세 명을 생성합니다. 각자 고유한 system prompt와 출력 형태를 받고, 업무에 필요한 tool만 받습니다. researcher는 web search를 받고, 사례 선정자는 로컬 사례 모음만 읽을 수 있으며, pricing modeler는 `pricing_rules.md`와 seat 수만 봅니다. 역할별로 tool 범위를 지정하면 가격 담당자가 web에서 경쟁사의 수치를 가져오는 일을 막고, 전체 사례 모음을 coordinator의 context 밖에 유지할 수 있습니다.


In [ ]:
def make_agent(name, description, system, tools):
    a = client.beta.agents.create(
        name=name,
        description=description,
        # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
        metadata={"anthropic_cookbook": "claude-cookbooks/cma-coordinate-specialist-team"},
        model=MODEL,
        system=system,
        tools=tools,
        betas=BETAS,
    )
    print(f"{name}: {a.id}")
    return a.id


prospect_researcher = make_agent(
    "prospect_researcher",
    "Researches what companies in a given industry segment and size tier typically prioritize.",
    """Given a prospect's industry and size, use web search to find:
- What companies in that segment typically list as strategic priorities
- Recent trends or pressures in that industry
- Common operational pain points at that scale
Return via send_to_parent: {"priorities": [...], "recent_moves": [...], "pain_points": [...], "sources": [...]}""",
    [
        {
            "type": "agent_toolset_20260401",
            "configs": [{"name": "web_search"}, {"name": "web_fetch"}],
        }
    ],
)

case_study_picker = make_agent(
    "case_study_picker",
    "Selects the two most relevant case studies from the library for a given prospect profile.",
    """The case study library is in /mnt/user-data/case_studies/. Each file is one customer story.
You will be given a prospect's industry, size, and top priorities. Read the library, score each study on relevance, and pick the two best matches.
Return via send_to_parent: {"picks": [{"file": ..., "customer": ..., "why_relevant": ...}, ...]}""",
    [{"type": "agent_toolset_20260401"}],
)

pricing_modeler = make_agent(
    "pricing_modeler",
    "Builds two or three pricing options for a prospect based on seat count and expected usage.",
    """Pricing rules are in /mnt/user-data/pricing_rules.md. Given a prospect's estimated seat count and usage tier, build:
- a conservative option (annual commit, lower per-seat)
- a flexible option (monthly, higher per-seat)
- if seat count > 500, an enterprise option with a platform fee
Show the first-year total for each. Return via send_to_parent: {"options": [{"name": ..., "structure": ..., "year_one_total": ...}, ...]}""",
    [{"type": "agent_toolset_20260401"}],
)

## 3. 팀에 작업 자료 제공하기

librarian은 선택할 사례 모음이 필요합니다. 의료, 제조, 물류, 소매, fintech, 공공 부문에 걸친 짧은 사례 일곱 개를 제공하여 잠재 고객에 맞는 두 개를 실제로 고르는 과정을 확인할 수 있도록 하겠습니다.


In [ ]:
CASE_STUDIES = [
    {
        "slug": "stclair_health",
        "title": "St. Clair Health",
        "industry": "regional hospital network",
        "employees": 6200,
        "summary": """Challenge: credentialing and prior-auth workflows spread across 11 systems.
Result with Northstar: consolidated to 3 automated workflows; prior-auth turnaround down 58%; $1.9M annual labor savings.""",
    },
    {
        "slug": "blueridge_health_plan",
        "title": "BlueRidge Health Plan",
        "industry": "regional payer",
        "employees": 2800,
        "summary": """Challenge: claims-adjudication exceptions queued in email; 19% required manual rework.
Result with Northstar: exception routing automated end-to-end; rework rate down to 6%; 11-day faster average claim resolution.""",
    },
    {
        "slug": "calder_mfg",
        "title": "Calder Manufacturing",
        "industry": "industrial",
        "employees": 3100,
        "summary": """Challenge: purchase-order approvals averaging 9 days.
Result with Northstar: PO cycle time cut to 2.1 days; 14% reduction in maverick spend.""",
    },
    {
        "slug": "northwind",
        "title": "Northwind Logistics",
        "industry": "3PL",
        "employees": 4400,
        "summary": """Challenge: carrier-onboarding paperwork took 3 weeks per carrier.
Result with Northstar: onboarding down to 4 days; 22% more carriers activated in Q1.""",
    },
    {
        "slug": "harborview_retail",
        "title": "Harborview Retail Group",
        "industry": "specialty retail",
        "employees": 5600,
        "summary": """Challenge: store-level inventory exceptions handled by regional managers over Slack and spreadsheets.
Result with Northstar: exception triage automated across 140 stores; stockout incidents down 31%.""",
    },
    {
        "slug": "aperture_fintech",
        "title": "Aperture Payments",
        "industry": "fintech",
        "employees": 1900,
        "summary": """Challenge: KYC and merchant-onboarding reviews averaging 6 business days.
Result with Northstar: review SLA cut to 36 hours; onboarding throughput up 2.4x with the same team.""",
    },
    {
        "slug": "summit_county",
        "title": "Summit County Government",
        "industry": "public sector",
        "employees": 3700,
        "summary": """Challenge: building-permit applications routed through five departments by paper packet.
Result with Northstar: single digital intake with parallel department review; median permit time 41 to 17 days.""",
    },
]

### 제품 및 가격 자료

coordinator가 "How we help" 절을 작성할 때 읽는 한 페이지 제품 소개 자료와, modeler가 선택지를 만드는 데 사용하는 가격 규칙 파일도 제공합니다.


In [ ]:
PRODUCT = """# Northstar Platform — One-Pager
Northstar is a workflow automation platform for mid-market operations teams.
Core capabilities: visual process builder, 200+ SaaS connectors, role-based approvals, SOC 2 Type II.
Typical results: 40-60% reduction in manual ticket handling, 3-week time-to-first-workflow."""

PRICING = """# Pricing Rules (internal)
- Per-seat list: $65/mo (monthly) or $52/mo (annual commit).
- Usage tiers: light = 1.0x, standard = 1.15x, heavy = 1.30x multiplier on per-seat.
- Enterprise (>500 seats): add $48,000/yr platform fee, per-seat drops to $44/mo annual.
- All options include onboarding; enterprise includes a named CSM."""

### coordinator 연결하고 세션 시작하기

이제 environment를 생성하고, 파일 아홉 개를 업로드한 뒤, specialist 세 명으로 구성된 `multiagent` roster를 가진 coordinator를 생성하겠습니다. 각 항목은 자체 model, prompt, toolset을 갖춘 완전한 agent이므로 역할별로 model 등급을 혼합할 수도 있습니다.

roster에는 agent가 아닌 항목도 하나 있습니다. `{"type": "advisor", "model": ...}`는 coordinator가 turn 도중 자문할 수 있는 더 강력한 model을 지정합니다. tool이 없고 생성하거나 message를 보낼 수 없습니다. coordinator가 `advisor` tool을 호출하면 플랫폼이 해당 model에 지금까지의 대화를 보여 주고 조언이 turn으로 돌아옵니다. 여기서는 작성 전에 사례 선택과 가격 제시 방식이 적절한지 점검합니다. 전체 패턴은 [`CMA_consult_an_advisor.ipynb`](CMA_consult_an_advisor.ipynb)에 있습니다.


In [ ]:
env = client.beta.environments.create(
    name="proposal-meridian",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
)

resources = []


def mount(path, content):
    f = client.beta.files.upload(file=(os.path.basename(path), content.encode(), "text/plain"))
    resources.append({"type": "file", "file_id": f.id, "mount_path": path})


for cs in CASE_STUDIES:
    body = f"# {cs['title']} ({cs['industry']}, {cs['employees']:,} employees)\n{cs['summary']}"
    mount(f"/mnt/user-data/case_studies/{cs['slug']}.md", body)
mount("/mnt/user-data/product_one_pager.md", PRODUCT)
mount("/mnt/user-data/pricing_rules.md", PRICING)

coordinator = client.beta.agents.create(
    name="Proposal Writer",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-coordinate-specialist-team"},
    model=MODEL,
    system="""You assemble tailored sales proposals.
Given a prospect name and basic profile:
1. Send the prospect's industry and size to prospect_researcher.
2. Send the prospect's industry, size, and (once the researcher reports back) their priorities to case_study_picker.
3. Send the seat count and usage tier to pricing_modeler.
4. Before writing, consult your advisor to check the case-study picks and the pricing framing against the prospect's priorities.
5. Read /mnt/user-data/product_one_pager.md, then write /mnt/session/outputs/proposal.md with sections:
   Executive summary (tied to their priorities), How we help (from the one-pager),
   Proof (the two case studies), Investment (the pricing options), Next steps.
Keep it to two pages.""",
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "coordinator",
        "agents": [
            prospect_researcher,
            case_study_picker,
            pricing_modeler,
            # coordinator가 turn 도중 자문할 수 있는 더 강력한 model
            {"type": "advisor", "model": ADVISOR_MODEL},
        ],
    },
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": coordinator.id, "version": coordinator.version},
    environment_id=env.id,
    resources=resources,
    title="Proposal: Meridian Health",
    betas=BETAS,
)
print(f"Session {session.id} ready with {len(resources)} files mounted")

## 4. 제안서 작성 시작하기

잠재 고객 profile을 보내고 coordinator가 일하는 모습을 살펴보겠습니다. researcher와 pricing modeler를 병렬로 시작한 다음, researcher의 조사 결과가 돌아오면 사례 선정자를 실행합니다. 선정자가 관련성을 평가하려면 그 우선순위가 필요하기 때문입니다. 작성하기 전 advisor에 자문하며, 이는 stream에서 짧게 유지되는 `anthropic.advisor` thread로 나타납니다.


In [ ]:
PROSPECT = {
    "name": "Meridian Health",
    "industry": "regional healthcare system",
    "employees": 8500,
    "estimated_seats": 600,
    "usage_tier": "heavy",
}

client.beta.sessions.events.send(
    session.id,
    betas=BETAS,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": f"Build a proposal for {PROSPECT['name']}, a {PROSPECT['industry']} with "
                    f"~{PROSPECT['employees']} employees. Estimate {PROSPECT['estimated_seats']} seats "
                    f"at {PROSPECT['usage_tier']} usage. Write to /mnt/session/outputs/proposal.md.",
                }
            ],
        }
    ],
)

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "session.thread_created":
            print(f"[spawn] {ev.agent_name}")
        elif ev.type == "agent.thread_message_received":
            if ev.from_agent_name == "anthropic.advisor":
                print("[advice] advisor guidance received")
            else:
                print(f"[report] {ev.from_agent_name} returned")
        elif ev.type == "session.status_idle":
            print("[done]")
            break

### 각 팀원이 돌려준 내용

완성된 제안서를 보기 전에 원본 `send_to_parent` payload 세 개와 advisor의 조언을 출력하겠습니다. 각 subagent는 자신의 context에서 자신에게 주어진 tool만 사용하여 실행되므로 세 보고서는 서로 상당히 다릅니다. advisor의 block은 coordinator가 작성 전에 요청한 적절성 점검입니다.


In [ ]:
def text_of(content):
    return "".join(b.text for b in content if b.type == "text")


for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if ev.type == "agent.thread_message_received":
        body = text_of(ev.content) or "[redacted]"  # advisor 내용은 공개되지 않을 수 있습니다
        label = (
            "advisor guidance"
            if ev.from_agent_name == "anthropic.advisor"
            else f"send_to_parent from {ev.from_agent_name}"
        )
        print(f"━━━ {label} ({len(body)} chars) ━━━")
        print(body[:1200] + (f"\n…[{len(body) - 1200} more chars]" if len(body) > 1200 else ""))
        print()

## 5. 제안서 읽기

마지막으로 완성된 제안서를 가져오겠습니다. coordinator가 `write` tool로 `proposal.md`에 작성했으므로 log에서 해당 event를 찾아 생성한 절을 확인합니다. 전체 문서를 읽으려면 `proposal` 자체를 출력하세요.


In [ ]:
proposal = ""
for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if (
        ev.type == "agent.tool_use"
        and ev.name == "write"
        and ev.input["file_path"].endswith("proposal.md")
    ):
        proposal = ev.input["content"]
        break

# 전체 제안서 대신 절 구조를 보여 줍니다.
for line in proposal.splitlines():
    if line.startswith("#"):
        print(line)

## agent 하나 대신 subagent 세 명을 사용하는 이유

tool 세 개를 모두 가진 단일 agent도 이 제안서를 작성할 수 있는데 왜 나누는 걸까요? 각 역할에 고유한 tool 범위를 지정하면 pricing modeler는 규칙 파일만 가지므로 web에서 경쟁사의 정가를 가져올 수 없습니다. 사례 선정자는 여기서는 파일 일곱 개를 읽지만 production에서는 수백 개를 읽게 되며, 그 분량은 coordinator 대신 subagent의 context에 머무릅니다. 또한 coordinator는 specialist 작업을 직접 수행하지 않으면서 순서와 인계를 결정할 수 있습니다.

multi-agent 조율에 관한 자세한 내용은 [Managed Agents 문서](https://platform.claude.com/docs/en/managed-agents/multi-agent)를 참고하세요.

